In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd

from docs.scripts.data_utils import sort_participant_data, build_participants_html


### Step 1: Read in the data
First load the data (your participant list csv file).

### IMPORTANT: MAKE SURE NO PERSONAL DATA IS IN THAT FILE

Then, check which columns are in the file, and select the ones you want to keep.
In this version, we have a very simplified column name structure, with just "Name", "Surname", "Affiliation", and "Attendance".

Finally, apply the transformation function to the attendance column (if needed), combine the name columns into a single "Participant" column, sort by surname, add a numbering column, and convert to an HTML table.

In [ ]:
# Load the participant list CSV file
filename = 'participant_list_web.csv'
filepath = f'../input_data/{filename}'
# Confirm which columns are in the file
# This will help you understand the structure of your data
pd.read_csv(filepath).columns

# Specify the columns to keep from the CSV file
columns_to_keep = ['Name',
                   'Surname',
                   'Affiliation',
                   'Attendance']

# Read the data file
data = pd.read_csv(filepath, usecols=columns_to_keep)

### Step 2: Sort the data

In [ ]:
data_sorted = sort_participant_data(data)

# Uncomment to confirm the sorting as this is going to be used in the HTML output
#data_sorted

### Step 3: Build the HTML page with the participant list

In [ ]:

html_content = build_participants_html(
    data_sorted,
    images=["assets/images/collaborations_bar.jpg", "assets/images/rank_bar.jpg"],
    menu_items=[("Statistics", "#statistics"), ("Participants", "#participants")]
)

print(html_content)

# Make histogram of collaborations

In [ ]:
# Read the data file
df = pd.read_csv(file_path, usecols=columns_to_keep)
collaboration_list = [str(i).split(',') for i in df['Are you a member of any cosmology collaborations (e.g., LSST DESC, Euclid, Roman)?\n\nTick all that apply.']]
# combine all the lists into one list
collaboration_list = [item.strip() for sublist in collaboration_list for item in sublist]

In [ ]:


def two_color_cmap(hex1, hex2, n=256, name='two_color'):
    """
    Create a linear colormap that blends from hex1 → hex2.

    Parameters
    ----------
    hex1, hex2 : str
        The start and end colors, e.g. "#FF0000", "#0000FF".
    n : int
        Number of discrete steps in the colormap (default 256).
    name : str
        Name for the colormap.

    Returns
    -------
    cmap : LinearSegmentedColormap
    """
    return LinearSegmentedColormap.from_list(name, [hex1, hex2], N=n)


# -- example usage --
cmap = two_color_cmap('#0C388A', '#002772')  # tomato → dodgerblue

In [ ]:
fig = plt.figure(figsize=(18, 5))
# sort histogram by frequency
collaboration_list.sort()
# remove nan
collaboration_list = [x for x in collaboration_list if str(x) != 'nan' and str(x)!='(applied to become LSST DESC member)' and str(x)!='No']

from matplotlib import cm

counts = pd.Series(collaboration_list).value_counts()
labels, values = counts.index, counts.values

# — plot setup —
fig, ax = plt.subplots(figsize=(18,5))
bars = ax.bar(labels, values)

# — gradient‐fill function —
def gradient_fill(bars, cmap_name="Blues"):
    cmap = cm.get_cmap(cmap_name)
    # Create a vertical 256-step gradient: shape (256,1)
    grad = np.linspace(0, 1, 256)[:, None]
    # (optional) repeat to two columns so interpolation is smoother:
    grad = np.repeat(grad, 2, axis=1)

    # remember limits
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()

    for bar in bars:
        bar.set_facecolor("none")
        x, y = bar.get_xy()
        w, h = bar.get_width(), bar.get_height()
        ax.imshow(
            grad,
            extent=[x, x + w, y, y + h],
            aspect="auto",
            cmap=cmap,
            origin="lower",   # <-- this makes row[0] sit at the bottom of the bar
            zorder=0,
        )

    # restore limits so the axes don't auto-scale to the image
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)

# — apply gradient to bars —
gradient_fill(bars, cmap_name=cmap)  # or "Blues", "viridis", etc.

# — annotate counts on top of bars —
ax.bar_label(bars, fmt="%d", padding=3, fontsize=12)

# — x‐axis styling —
ax.set_xlabel("Collaboration", fontsize=18)
ax.tick_params(axis="x", labelsize=14)

# — remove y‐axis entirely —
ax.tick_params(axis="y", left=False, labelleft=False)
ax.spines["left"].set_visible(False)

# — tidy up other spines if you like —
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


plt.savefig('assets/images/collaborations_bar.jpg', bbox_inches='tight', dpi=300)

In [ ]:
fig = plt.figure(figsize=(18, 5))
# sort histogram by frequency
collaboration_list.sort()
# remove nan
collaboration_list = [x for x in collaboration_list if str(x) != 'nan' and str(x)!='(applied to become LSST DESC member)' and str(x)!='No']

# make bar plot
bar = plt.bar(*zip(*dict(pd.Series(collaboration_list).value_counts()).items()))
# make each bar color a gradient of blues and purples
print(collaboration_list)
def gradientbars(bars):
    grad = np.atleast_2d(np.linspace(0,1,256)).T
    ax = bars[0].axes
    lim = ax.get_xlim()+ax.get_ylim()
    for bar in bars:
        bar.set_zorder(1)
        bar.set_facecolor("none")
        x,y = bar.get_xy()
        w, h = bar.get_width(), bar.get_height()
        ax.imshow(grad, extent=[x,x+w,y,y+h], aspect="auto", zorder=0, cmap=cmap, vmin=0, vmax=2, alpha=.8)
        
        ax.bar_label(bars, fmt="%d", padding=2, fontsize=12)
    ax.axis(lim)
gradientbars(bar)

plt.xlabel('Collaboration', fontsize=18)
plt.xticks(rotation=0, ha='center', fontsize=16)
# hide axis lines
plt.box(False)
plt.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=True)

# save
plt.savefig('assets/images/collaborations_bar.jpg', bbox_inches='tight', dpi=300)

Update html file

In [ ]:
# update html content
# Build the HTML content for your participants page with the table
html_content = f"""---
layout: default
title: Participants
order: 5
---

<p align="center">
  <img src="assets/images/collaborations_bar.jpg" alt="Participant Collaborations" width="1000">
</p>
<br>
<br>

{html_table}
"""

# Print the HTML content to the console or write it to an HTML file
print(html_content)